## ***States and Schemas handling for the Research Assistant***

In [28]:
from typing import Annotated, Literal
# Reuse the application function instead of defining it again here.
from aegis.clarification import check_clarity

from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    StringConstraints,
    model_validator,
)

NonEmptyText = Annotated[
    str,
    StringConstraints(strip_whitespace=True, min_length=1),
]

# Import application code instead of maintaining separate notebook copies.
from aegis.schemas import (
    NonEmptyText,
    ResearchScope,
    ResearchRequest,
    ClarificationQuestion,
    ClarificationResult,
    SourceRecord,
    EvidenceItem,
    ResearchFinding,
    ResearchBundle,
)
from aegis.clarification import check_clarity

# Utilities used by the notebook's examples and validation checks.
from copy import deepcopy
from datetime import datetime, timezone

from pydantic import ValidationError
 

# class ResearchScope(BaseModel):
#     model_config = ConfigDict(extra="forbid")

#     time_period: NonEmptyText | None = None
#     coverage: NonEmptyText | None = None


# class ResearchRequest(BaseModel):
#     model_config = ConfigDict(extra="forbid")
#     objective: NonEmptyText
#     # Prototype limits to keep research runs manageable.
#     subjects: list[NonEmptyText] = Field(
#         min_length=1,
#         max_length=5,
#     )
#     # Empty means the factors still need clarification or proposal.
#     factors: list[NonEmptyText] = Field(
#         default_factory=list,
#         max_length=10,
#     )
#     mode: Literal["research", "compare", "rank"] = "research"
#     scope: ResearchScope = Field(default_factory=ResearchScope)
#     constraints: list[NonEmptyText] = Field(default_factory=list)
#     ranking_basis: NonEmptyText | None = None


#     @model_validator(mode="after")
#     def validate_consistency(self):
#         subject_keys = [name.casefold() for name in self.subjects]

#         if len(subject_keys) != len(set(subject_keys)):
#             raise ValueError("Subjects must be unique")

#         factor_keys = [name.casefold() for name in self.factors]

#         if len(factor_keys) != len(set(factor_keys)):
#             raise ValueError("Factors must be unique")

#         if self.mode in {"compare", "rank"} and len(self.subjects) < 2:
#             raise ValueError(
#                 "Comparison or ranking requires at least two subjects"
#             )

#         return self

In [ ]:
# class ClarificationQuestion(BaseModel):
#     model_config = ConfigDict(extra="forbid")

#     field: NonEmptyText
#     question: NonEmptyText
#     reason: NonEmptyText


# class ClarificationResult(BaseModel):
#     model_config = ConfigDict(extra="forbid")

#     questions: list[ClarificationQuestion] = Field(
#         default_factory=list
#     )

#     @property
#     def needs_clarification(self) -> bool:
#         return bool(self.questions)

In [ ]:
# def check_clarity(
#     request: ResearchRequest,
# ) -> ClarificationResult:
#     questions = []

#     if not request.factors:
#         questions.append(
#             ClarificationQuestion(
#                 field="factors",
#                 question=(
#                     "Which aspects should the research focus on? "
#                     "You can also ask Aegis to propose some for approval."
#                 ),
#                 reason=(
#                     "The selected factors determine what evidence "
#                     "we need to collect."
#                 ),
#             )
#         )

#     if request.scope.time_period is None:
#         questions.append(
#             ClarificationQuestion(
#                 field="scope.time_period",
#                 question=(
#                     "What period should the research cover? "
#                     "Specify a date range, an as-of date, "
#                     "or say that time is not relevant."
#                 ),
#                 reason=(
#                     "Evidence from different periods may not "
#                     "support a fair comparison."
#                 ),
#             )
#         )

#     if request.scope.coverage is None:
#         questions.append(
#             ClarificationQuestion(
#                 field="scope.coverage",
#                 question=(
#                     "What context or boundaries should apply? "
#                     "For example: competitions, geography, use case, "
#                     "or a general overview."
#                 ),
#                 reason=(
#                     "Coverage determines which evidence belongs "
#                     "in the research."
#                 ),
#             )
#         )

#     if request.mode == "rank" and request.ranking_basis is None:
#         questions.append(
#             ClarificationQuestion(
#                 field="ranking_basis",
#                 question=(
#                     "How should we decide the ranking? "
#                     "State your priorities or provide factor weights."
#                 ),
#                 reason=(
#                     "Choosing a winner requires an explicit "
#                     "decision rule."
#                 ),
#             )
#         )

#     return ClarificationResult(questions=questions)

In [29]:
# Test cell - 1

football_request = ResearchRequest(
    objective=(
        "Compare Lionel Messi and Cristiano Ronaldo "
        "on scoring and creativity."
    ),
    subjects=["Lionel Messi", "Cristiano Ronaldo"],
    factors=["Scoring", "Creativity"],
    mode="compare",
    scope=ResearchScope(
        time_period="2022–2025 calendar years, inclusive",
        coverage="Senior club and international competitive matches",
    ),
    constraints=[
        "Distinguish totals from per-90-minute statistics",
        "Explain differences in competition context",
        "Do not invent unavailable statistics",
        "Prioritize creativity; use scoring as a tiebreaker."
    ],
)

print(football_request.model_dump_json(indent=2))

{
  "objective": "Compare Lionel Messi and Cristiano Ronaldo on scoring and creativity.",
  "subjects": [
    "Lionel Messi",
    "Cristiano Ronaldo"
  ],
  "factors": [
    "Scoring",
    "Creativity"
  ],
  "mode": "compare",
  "scope": {
    "time_period": "2022–2025 calendar years, inclusive",
    "coverage": "Senior club and international competitive matches"
  },
  "constraints": [
    "Distinguish totals from per-90-minute statistics",
    "Explain differences in competition context",
    "Do not invent unavailable statistics",
    "Prioritize creativity; use scoring as a tiebreaker."
  ],
  "ranking_basis": null
}


In [30]:
# Test Cell - 2

platform_request = ResearchRequest(
    objective="Compare data platforms for a small analytics team.",
    subjects=["Snowflake", "Databricks", "Microsoft Fabric"],
    factors=["Operating cost", "Ease of maintenance", "Security"],
    mode="compare",
    scope=ResearchScope(
        coverage="A small team running daily batch analytics",
    ),
)

print(platform_request.model_dump_json(indent=2))

{
  "objective": "Compare data platforms for a small analytics team.",
  "subjects": [
    "Snowflake",
    "Databricks",
    "Microsoft Fabric"
  ],
  "factors": [
    "Operating cost",
    "Ease of maintenance",
    "Security"
  ],
  "mode": "compare",
  "scope": {
    "time_period": null,
    "coverage": "A small team running daily batch analytics"
  },
  "constraints": [],
  "ranking_basis": null
}


In [31]:
# Test Cell - 3 : Failure Cell

from pydantic import ValidationError

try:
    ResearchRequest(
        objective="Compare footballers.",
        subjects=["Lionel Messi"],
        mode="compare",
    )
except ValidationError as exc:
    for error in exc.errors(include_input=False):
        print(error["msg"])
else:
    raise AssertionError("A one-subject comparison was accepted")

Value error, Comparison or ranking requires at least two subjects


In [18]:
incomplete_request = ResearchRequest(
    objective="Compare Messi and Ronaldo.",
    subjects=["Lionel Messi", "Cristiano Ronaldo"],
    mode="compare",
)

result = check_clarity(incomplete_request)

print("Needs clarification:", result.needs_clarification)

for item in result.questions:
    print(f"\nField: {item.field}")
    print(f"Question: {item.question}")
    print(f"Why: {item.reason}")

Needs clarification: True

Field: factors
Question: Which aspects should the research focus on? You can also ask Aegis to propose some for approval.
Why: The selected factors determine what evidence we need to collect.

Field: scope.time_period
Question: What period should the research cover? Specify a date range, an as-of date, or say that time is not relevant.
Why: Evidence from different periods may not support a fair comparison.

Field: scope.coverage
Question: What context or boundaries should apply? For example: competitions, geography, use case, or a general overview.
Why: Coverage determines which evidence belongs in the research.


In [32]:
updated_payload = incomplete_request.model_dump()

updated_payload["factors"] = ["Scoring", "Creativity"]
updated_payload["scope"] = {
    "time_period": "2022–2025 calendar years, inclusive",
    "coverage": (
        "Senior club and international competitive matches"
    ),
}

clarified_request = ResearchRequest.model_validate(updated_payload)

clarified_result = check_clarity(clarified_request)

print("Needs clarification:", clarified_result.needs_clarification)
print("Questions remaining:", len(clarified_result.questions))

Needs clarification: False
Questions remaining: 0


In [33]:
ranking_payload = clarified_request.model_dump()
ranking_payload["mode"] = "rank"

ranking_request = ResearchRequest.model_validate(ranking_payload)
ranking_result = check_clarity(ranking_request)

assert [
    item.field for item in ranking_result.questions
] == ["ranking_basis"]

ranking_payload["ranking_basis"] = (
    "Prioritize creativity; use scoring as a tiebreaker."
)

ranking_request = ResearchRequest.model_validate(ranking_payload)

assert not check_clarity(ranking_request).needs_clarification

print("Comparison and ranking clarification checks passed.")

Comparison and ranking clarification checks passed.


## ***Source | Evidence | Research***

In [ ]:
# from datetime import date, datetime, timezone
# from typing import Literal

# from pydantic import (
#     AwareDatetime,
#     BaseModel,
#     ConfigDict,
#     Field,
#     HttpUrl,
#     ValidationError,
#     model_validator,
# )


# class SourceRecord(BaseModel):
#     """Describe material retrieved by a tool or loaded from a dataset."""

#     model_config = ConfigDict(extra="forbid")

#     # Created by our retrieval code, not invented by the LLM.
#     source_id: NonEmptyText
#     title: NonEmptyText

#     # The source type determines which location field is required.
#     kind: Literal["web", "dataset"]
#     url: HttpUrl | None = None
#     dataset_ref: NonEmptyText | None = None

#     # Unknown publication dates stay unknown.
#     published_on: date | None = None

#     # Explicit timestamp supplied when the material is retrieved.
#     retrieved_at: AwareDatetime

#     # Keeps synthetic examples distinguishable from real research.
#     is_test_data: bool = False

#     @model_validator(mode="after")
#     def validate_location(self):
#         """Require the correct location for the selected source type."""

#         if self.kind == "web":
#             if self.url is None or self.dataset_ref is not None:
#                 raise ValueError(
#                     "Web sources require a URL and no dataset_ref"
#                 )

#         if self.kind == "dataset":
#             if self.dataset_ref is None or self.url is not None:
#                 raise ValueError(
#                     "Dataset sources require dataset_ref and no URL"
#                 )

#         return self

In [ ]:
# class EvidenceItem(BaseModel):
#     """Store an extracted passage or dataset row for later verification."""

#     model_config = ConfigDict(extra="forbid")

#     evidence_id: NonEmptyText

#     # Acts like a foreign key to SourceRecord.source_id.
#     source_id: NonEmptyText

#     # These remain general: a player, product, platform, or another subject.
#     subject: NonEmptyText
#     factor: NonEmptyText

#     # Preserve the retrieved text or a faithful representation of a row.
#     # This is evidence material, not the model's interpretation.
#     excerpt: NonEmptyText

#     # Examples: "Table 2, row 7", "page 4", or a section heading.
#     locator: NonEmptyText

#     # Preserve contextual limits so incompatible statistics aren't mixed.
#     time_period: NonEmptyText | None = None
#     coverage: NonEmptyText | None = None

In [ ]:
# class ResearchFinding(BaseModel):
#     """Represent a sourced claim awaiting independent verification."""

#     model_config = ConfigDict(extra="forbid")

#     finding_id: NonEmptyText
#     subject: NonEmptyText
#     factor: NonEmptyText
#     claim: NonEmptyText

#     # Distinguish a reported observation from an interpretation.
#     claim_type: Literal["reported_fact", "interpretation"]

#     # A finding must reference at least one evidence item.
#     evidence_ids: list[NonEmptyText] = Field(min_length=1)

#     # Record qualifications rather than hiding them in the final report.
#     limitations: list[NonEmptyText] = Field(default_factory=list)

#     # This stage cannot mark its own findings as verified.
#     verification_status: Literal["unverified"] = "unverified"

In [ ]:
# class ResearchBundle(BaseModel):
#     """Group research artifacts and check their reference integrity."""

#     model_config = ConfigDict(extra="forbid")

#     sources: list[SourceRecord] = Field(default_factory=list)
#     evidence: list[EvidenceItem] = Field(default_factory=list)
#     findings: list[ResearchFinding] = Field(default_factory=list)

#     @model_validator(mode="after")
#     def validate_references(self):
#         """Reject duplicate IDs and references to missing records."""

#         source_ids = [item.source_id for item in self.sources]
#         evidence_ids = [item.evidence_id for item in self.evidence]
#         finding_ids = [item.finding_id for item in self.findings]

#         # Duplicate IDs would make citation resolution ambiguous.
#         for label, identifiers in (
#             ("source", source_ids),
#             ("evidence", evidence_ids),
#             ("finding", finding_ids),
#         ):
#             if len(identifiers) != len(set(identifiers)):
#                 raise ValueError(f"Duplicate {label} IDs")

#         known_sources = set(source_ids)
#         known_evidence = set(evidence_ids)

#         # Every evidence item must point to an existing source.
#         for item in self.evidence:
#             if item.source_id not in known_sources:
#                 raise ValueError(
#                     f"Evidence {item.evidence_id} references "
#                     f"unknown source {item.source_id}"
#                 )

#         # Every finding must point to existing, distinct evidence items.
#         for finding in self.findings:
#             if len(finding.evidence_ids) != len(set(finding.evidence_ids)):
#                 raise ValueError(
#                     f"Finding {finding.finding_id} repeats evidence IDs"
#                 )

#             missing = set(finding.evidence_ids) - known_evidence

#             if missing:
#                 raise ValueError(
#                     f"Finding {finding.finding_id} references "
#                     f"unknown evidence: {sorted(missing)}"
#                 )

#         return self

In [34]:
#  Test Cell - 4

# Create a synthetic source record.
# No file is read here; the dataset reference is just a test identifier.
source = SourceRecord(
    source_id="src_demo_001",
    title="Synthetic football statistics for schema testing",
    kind="dataset",
    dataset_ref="fixture://football_demo_v1",
    retrieved_at=datetime.now(timezone.utc),
    is_test_data=True,
)

# Represent one row from that fictional dataset.
evidence = EvidenceItem(
    evidence_id="ev_demo_001",
    source_id=source.source_id,
    subject="Player A",
    factor="Scoring",
    excerpt="Player A | appearances: 10 | goals: 8",
    locator="players table, row player_a",
    time_period="Demo season",
    coverage="Fictional competition",
)

# Make a reported claim supported by that row.
finding = ResearchFinding(
    finding_id="finding_demo_001",
    subject="Player A",
    factor="Scoring",
    claim="The demo dataset records 8 goals for Player A.",
    claim_type="reported_fact",
    evidence_ids=[evidence.evidence_id],
    limitations=["Synthetic data; not suitable for real comparisons."],
)

# Validate all objects and their links together.
bundle = ResearchBundle(
    sources=[source],
    evidence=[evidence],
    findings=[finding],
)

print(bundle.model_dump_json(indent=2))

{
  "sources": [
    {
      "source_id": "src_demo_001",
      "title": "Synthetic football statistics for schema testing",
      "kind": "dataset",
      "url": null,
      "dataset_ref": "fixture://football_demo_v1",
      "published_on": null,
      "retrieved_at": "2026-11-17T16:07:44.052507Z",
      "is_test_data": true
    }
  ],
  "evidence": [
    {
      "evidence_id": "ev_demo_001",
      "source_id": "src_demo_001",
      "subject": "Player A",
      "factor": "Scoring",
      "excerpt": "Player A | appearances: 10 | goals: 8",
      "locator": "players table, row player_a",
      "time_period": "Demo season",
      "coverage": "Fictional competition"
    }
  ],
  "findings": [
    {
      "finding_id": "finding_demo_001",
      "subject": "Player A",
      "factor": "Scoring",
      "claim": "The demo dataset records 8 goals for Player A.",
      "claim_type": "reported_fact",
      "evidence_ids": [
        "ev_demo_001"
      ],
      "limitations": [
        "Synthetic 

In [35]:
# Reject broken records

from copy import deepcopy

# Export a valid bundle into ordinary Python data.
valid_payload = bundle.model_dump(mode="json")


def expect_rejection(payload: dict, expected_message: str) -> None:
    """Confirm that invalid research data fails for the expected reason."""

    try:
        ResearchBundle.model_validate(payload)
    except ValidationError as exc:
        messages = " | ".join(
            error["msg"] for error in exc.errors(include_input=False)
        )
        assert expected_message in messages, messages
        print(f"PASS: {expected_message}")
    else:
        raise AssertionError("Invalid research data was accepted")


# Test 1: Evidence points to a source that doesn't exist.
bad_source = deepcopy(valid_payload)
bad_source["evidence"][0]["source_id"] = "src_missing"
expect_rejection(bad_source, "unknown source")


# Test 2: A finding cites evidence that doesn't exist.
bad_citation = deepcopy(valid_payload)
bad_citation["findings"][0]["evidence_ids"] = ["ev_missing"]
expect_rejection(bad_citation, "unknown evidence")


# Test 3: Two sources share an ID, making references ambiguous.
duplicate_source = deepcopy(valid_payload)
duplicate_source["sources"].append(
    deepcopy(duplicate_source["sources"][0])
)
expect_rejection(duplicate_source, "Duplicate source IDs")

PASS: unknown source
PASS: unknown evidence
PASS: Duplicate source IDs
